In [1]:
print('hello world')

hello world


In [3]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from sklearn.ensemble import RandomForestClassifier
import torch.nn as nn
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import threading

In [4]:
# Generate dataset
data = []
for _ in range(500):
    latitude = random.uniform(10, 40)
    longitude = random.uniform(60, 100)
    depth = random.uniform(1, 300)
    magnitude = round(random.uniform(2.0, 8.0), 1)

    if magnitude > 5:
        earthquake = 1
    else:
        earthquake = 0

    data.append([latitude, longitude, depth, magnitude, earthquake])

df = pd.DataFrame(data, columns=[
    "latitude", "longitude", "depth", "magnitude", "earthquake"
])

df.to_csv("earthquake.csv", index=False)
print("Dataset created ")

# Load and preprocess data
df = pd.read_csv("earthquake.csv")
X = df.drop("earthquake", axis=1)
y = df["earthquake"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

# Train ML model
ml_model = RandomForestClassifier()
ml_model.fit(X_train.numpy(), y_train.numpy())
ml_acc = ml_model.score(X_test.numpy(), y_test.numpy())
print("ML Accuracy:", ml_acc)

# Define Neural Network
class EarthquakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

# Train DL model
model = EarthquakeModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    dl_acc = (predicted == y_test).sum().item() / len(y_test)
print("DL Accuracy:", dl_acc)

# Prediction function
def predict_earthquake(lat, lon, depth, magnitude):
    import pandas as pd
    import torch

    data = pd.DataFrame([[lat, lon, depth, magnitude]],
                        columns=["latitude", "longitude", "depth", "magnitude"])

    data = scaler.transform(data)
    tensor = torch.tensor(data, dtype=torch.float32)

    with torch.no_grad():
        output = model(tensor)
        _, pred = torch.max(output, 1)

    return "Earthquake Likely" if pred.item() == 1 else "No Earthquake"

# Tkinter GUI
class EarthquakePredictorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Earthquake Prediction System")
        self.root.geometry("1200x800")
        
        # Title
        title_label = tk.Label(root, text="🌍 Earthquake Prediction System", 
                              font=("Arial", 20, "bold"))
        title_label.pack(pady=10)
        
        # Accuracy info
        accuracy_frame = tk.Frame(root)
        accuracy_frame.pack(pady=5)
        tk.Label(accuracy_frame, text=f"ML Model Accuracy: {ml_acc:.3f}", 
                font=("Arial", 10), bg="#4CAF50", fg="white", padx=10, pady=5).pack(side=tk.LEFT, padx=10)
        tk.Label(accuracy_frame, text=f"DL Model Accuracy: {dl_acc:.3f}", 
                font=("Arial", 10), bg="#2196F3", fg="white", padx=10, pady=5).pack(side=tk.LEFT, padx=10)
        
        # Main frame
        main_frame = tk.Frame(root)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Left frame - Input and Predict
        left_frame = tk.Frame(main_frame)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 10))
        
        # Input frame
        input_frame = tk.LabelFrame(left_frame, text="Predict Earthquake", font=("Arial", 12, "bold"))
        input_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Input fields
        tk.Label(input_frame, text="Latitude (10-40):").grid(row=0, column=0, sticky=tk.W, padx=5, pady=8)
        self.lat_var = tk.StringVar(value="28.6")
        tk.Entry(input_frame, textvariable=self.lat_var, width=20).grid(row=0, column=1, padx=5, pady=8)
        
        tk.Label(input_frame, text="Longitude (60-100):").grid(row=1, column=0, sticky=tk.W, padx=5, pady=8)
        self.lon_var = tk.StringVar(value="77.2")
        tk.Entry(input_frame, textvariable=self.lon_var, width=20).grid(row=1, column=1, padx=5, pady=8)
        
        tk.Label(input_frame, text="Depth (km):").grid(row=2, column=0, sticky=tk.W, padx=5, pady=8)
        self.depth_var = tk.StringVar(value="50")
        tk.Entry(input_frame, textvariable=self.depth_var, width=20).grid(row=2, column=1, padx=5, pady=8)
        
        tk.Label(input_frame, text="Magnitude:").grid(row=3, column=0, sticky=tk.W, padx=5, pady=8)
        self.mag_var = tk.StringVar(value="6.5")
        tk.Entry(input_frame, textvariable=self.mag_var, width=20).grid(row=3, column=1, padx=5, pady=8)
        
        # Predict button
        predict_btn = tk.Button(input_frame, text="🔍 Predict Earthquake", command=self.predict_earthquake,
                               bg="#FF5722", fg="white", font=("Arial", 12, "bold"), width=20)
        predict_btn.grid(row=4, column=0, columnspan=2, pady=20)
        
        # Prediction history
        history_frame = tk.LabelFrame(left_frame, text="Recent Predictions", font=("Arial", 12, "bold"))
        history_frame.pack(fill=tk.BOTH, expand=True)
        
        self.history_listbox = tk.Listbox(history_frame, height=12, font=("Arial", 10))
        scrollbar = ttk.Scrollbar(history_frame, orient=tk.VERTICAL, command=self.history_listbox.yview)
        self.history_listbox.configure(yscrollcommand=scrollbar.set)
        
        self.history_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        clear_btn = tk.Button(history_frame, text="Clear History", command=self.clear_history,
                            bg="#f44336", fg="white")
        clear_btn.pack(pady=5)
        
        # Right frame - Dataset display
        right_frame = tk.Frame(main_frame)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Dataset frame
        dataset_frame = tk.LabelFrame(right_frame, text="Earthquake Dataset (500 records)", 
                                    font=("Arial", 12, "bold"))
        dataset_frame.pack(fill=tk.BOTH, expand=True)
        
        # Treeview for displaying data
        columns = ("Latitude", "Longitude", "Depth", "Magnitude", "Earthquake")
        self.tree = ttk.Treeview(dataset_frame, columns=columns, show="headings", height=20)
        
        # Define headings
        for col in columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=120)
        
        # Scrollbar for treeview
        tree_scrollbar = ttk.Scrollbar(dataset_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=tree_scrollbar.set)
        
        # Pack treeview and scrollbar
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        tree_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Load data button
        load_btn = tk.Button(dataset_frame, text="🔄 Refresh Dataset", command=self.load_data,
                           bg="#2196F3", fg="white", font=("Arial", 10, "bold"))
        load_btn.pack(pady=5)
        
        # Statistics frame
        stats_frame = tk.LabelFrame(right_frame, text="Dataset Statistics", font=("Arial", 12, "bold"))
        stats_frame.pack(fill=tk.X, pady=(5, 0))
        
        self.stats_label = tk.Label(stats_frame, text="", font=("Arial", 10), justify=tk.LEFT)
        self.stats_label.pack(pady=5)
        
        # Status label
        self.status_var = tk.StringVar(value="Ready - Enter seismic data to predict")
        status_label = tk.Label(right_frame, textvariable=self.status_var, 
                              relief=tk.SUNKEN, anchor=tk.W, bg="#f0f0f0")
        status_label.pack(side=tk.BOTTOM, fill=tk.X, pady=(5, 0))
        
        # Load initial data and stats
        self.load_data()
        self.update_stats()
    
    def load_data(self):
        """Load all earthquake records into the treeview"""
        # Clear existing data
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Load all records
        original_df = pd.read_csv("earthquake.csv")
        for index, row in original_df.iterrows():
            eq_status = "Yes" if row['earthquake'] == 1 else "No"
            self.tree.insert("", tk.END, values=(
                f"{row['latitude']:.2f}", f"{row['longitude']:.2f}", 
                f"{row['depth']:.1f}", f"{row['magnitude']:.1f}", eq_status
            ))
        self.status_var.set(f"✅ Loaded {len(original_df)} records")
    
    def update_stats(self):
        """Update dataset statistics"""
        df_stats = pd.read_csv("earthquake.csv")
        total = len(df_stats)
        earthquakes = len(df_stats[df_stats['earthquake'] == 1])
        avg_mag = df_stats['magnitude'].mean()
        max_mag = df_stats['magnitude'].max()
        
        stats_text = f"""
📊 Dataset Statistics:
- Total Records: {total}
- Earthquakes (>5.0): {earthquakes} ({earthquakes/total*100:.1f}%)
- Average Magnitude: {avg_mag:.2f}
- Maximum Magnitude: {max_mag:.1f}
- Average Depth: {df_stats['depth'].mean():.1f} km
        """
        self.stats_label.config(text=stats_text.strip())
    
    def predict_earthquake(self):
        """Make earthquake prediction"""
        try:
            lat = float(self.lat_var.get())
            lon = float(self.lon_var.get())
            depth = float(self.depth_var.get())
            magnitude = float(self.mag_var.get())
            
            if not (10 <= lat <= 40 and 60 <= lon <= 100):
                raise ValueError("Latitude must be 10-40, Longitude 60-100")
            if not (1 <= depth <= 300):
                raise ValueError("Depth must be 1-300 km")
            if not (2.0 <= magnitude <= 8.0):
                raise ValueError("Magnitude must be 2.0-8.0")
            
            prediction = predict_earthquake(lat, lon, depth, magnitude)
            
            # Add to history
            timestamp = pd.Timestamp.now().strftime("%H:%M:%S")
            history_entry = f"{timestamp} | Lat:{lat:.2f} Lon:{lon:.2f} D:{depth:.1f} M:{magnitude:.1f} → {prediction}"
            self.history_listbox.insert(0, history_entry)
            if self.history_listbox.size() > 10:
                self.history_listbox.delete(10)
            
            # Show prediction popup
            color = "#4CAF50" if "No" in prediction else "#FF5722"
            messagebox.showinfo("Prediction Result", 
                              f"🌍 Earthquake Prediction\n\n"
                              f"📍 Location: {lat:.2f}°, {lon:.2f}°\n"
                              f"📏 Depth: {depth:.1f} km\n"
                              f"⚡ Magnitude: {magnitude:.1f}\n"
                              f"\n🎯 **{prediction}**\n",
                              icon='info')
            
            self.status_var.set(f"🔍 Predicted: {prediction} (M:{magnitude:.1f})")
            
        except ValueError as e:
            messagebox.showerror("Input Error", f"Invalid input:\n{str(e)}")
        except Exception as e:
            messagebox.showerror("Prediction Error", f"Prediction failed:\n{str(e)}")
    
    def clear_history(self):
        """Clear prediction history"""
        self.history_listbox.delete(0, tk.END)

def main():
    root = tk.Tk()
    app = EarthquakePredictorApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()

Dataset created 
ML Accuracy: 1.0
Epoch: 0 Loss: 0.7136750817298889
Epoch: 10 Loss: 0.6773641705513
Epoch: 20 Loss: 0.6396735310554504
Epoch: 30 Loss: 0.5958296656608582
Epoch: 40 Loss: 0.5444484949111938
DL Accuracy: 0.78
